# V0.5 Resource Handle Lab

问题：pytest 输出几 MB，为什么不会把 Context 撑爆？

这个 notebook 逐格展示大输出如何进入 ResourceService，模型只看到 bounded preview。

## Mode

`deterministic`：Model decision = SCRIPTED，Kernel execution = REAL。

`real_model`：Model decision = REAL OpenAI-compatible，Kernel execution = REAL。优先使用 `AGENTKERNEL_LAB_LLM_*`，否则复用仓库本地 `.minicode/config.json`。不会展示 hidden chain-of-thought，也不会展示 API key。

In [ ]:
MODE = "deterministic"

from pathlib import Path
import sys

def find_agentkernel_root(start: Path) -> Path:
    for path in (start, *start.parents):
        if (path / "agentkernel").is_dir() and (path / "labs").is_dir():
            return path
    raise RuntimeError("Run this notebook from the AgentKernel repo root or the labs directory.")

REPO_ROOT = find_agentkernel_root(Path.cwd().resolve())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from labs import create_lab

lab = create_lab("v05", mode=MODE)


## Step 1: Setup

创建一个模拟的大 stdout。

In [ ]:
lab.setup()

## Step 2: Inspect raw output

先看原始输出规模，确认它不应该完整进入模型上下文。

In [ ]:
lab.show_large_output()

## Step 3: Externalize to ResourceService

Kernel 把完整 bytes 存为 artifact，只把 preview 和 ResourceHandle 返回给模型。

In [ ]:
lab.externalize_output()

## Step 4: Inspect model-visible request

观察模型看到的是 preview 和 handle metadata，而不是完整大输出。

In [ ]:
lab.show_model_request()

## Step 5: Ask deterministic or real model

模型可以解释 preview，但 handle 本身不是读取权限。

In [ ]:
lab.model_step()

## Step 6: Authorized read

拥有者通过 ResourceService 读取完整 artifact 的一小段。

In [ ]:
lab.authorized_read()

## Step 7: Unauthorized read

另一个 Agent 即使拿到 URI，也不能绕过 ResourceService 权限检查。

In [ ]:
lab.unauthorized_read()

## Summary

这个实验回答：大输出保真存储，Context 只承载 bounded preview。

In [ ]:
lab.summary()
lab.close()